## Reddit

#### Useful links:
- [Reddit API Documentation](https://www.reddit.com/dev/api)
- [Reddit Data API Wiki](https://support.reddithelp.com/hc/en-us/articles/16160319875092-Reddit-Data-API-Wiki)
- [Reddit for Researchers](https://www.reddit.com/r/reddit4researchers/)
- [Praw - Python client for Reddit](https://praw.readthedocs.io/en/stable/)
- [How to use Reddit API - Article](https://gcdi.commons.gc.cuny.edu/2024/11/01/web-scraping-with-python-and-the-reddit-api/#:~:text=Reddit%20has%20its%20own%20API,way%20to%20scrape%20Reddit%20data.)

In [ ]:
# requirements
%pip install pandas requests python-dotenv praw

In [1]:
import os
import json
import pandas as pd
import praw
import pprint
import glob
from time import sleep
from dotenv import load_dotenv
from datetime import datetime, timezone

In [2]:
# environment variables
load_dotenv()
REDDIT_CLIENT_ID = os.getenv("REDDIT_CLIENT_ID")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
REDDIT_USERAGENT = os.getenv("REDDIT_USERAGENT")
REDDIT_FILEPATH = os.getenv("REDDIT_FILEPATH")
FILEPATH = os.getenv("FILEPATH")

In [3]:
# API connection
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USERAGENT,
)

### SPECIAL CRITERIA

**SC1: Does the platform offer an API for collecting public user-generated content data?**

*This item verifies whether the platform provides an API with at least one endpoint for programmatically extracting public user-generated content to the users’ infrastructure. Public user-generated content is defined here as any publicly visible publication accessible by any platform user. The assessment should verify that the endpoint allows retrieval and storage of this content without requiring privileged or internal access beyond standard developer registration.*

In [5]:
# Getting the most recent entry of the r/all subreddit

sr = reddit.subreddit("all")

posts = sr.new(limit=2)
posts = list(posts)

post = posts[1]

print(f"Post ID: {post.id}")
print(f"Title: {post.title}")
print(f"Subreddit: {post.subreddit_name_prefixed}")
print(f"Created At: {post.created}")
print(f"Post permalink: https://reddit.com{post.permalink}")
print(f"Post URL: {post.url}")
print(f"Post Body: {post.selftext}")
print(f"Post Author: {post.author.name}")

Post ID: 1p6ooiq
Title: TRADING ALL! Lf offers
Subreddit: r/stealabrainrot
Created At: 1764105159.0
Post permalink: https://reddit.com/r/stealabrainrot/comments/1p6ooiq/trading_all_lf_offers/
Post URL: https://www.reddit.com/gallery/1p6ooiq
Post Body: mostly carpet spawns but offer away!
Post Author: Awkward_Sir3674


**SC2: Can the full scope of public content data be extracted through the platform’s API?** 

*This item verifies whether the platform enables programmatic discovery and extraction of data from the complete set of public user-generated content. The assessment should confirm that the API provides access to all types of public content on the platform, without exclusions or artificial restrictions that limit data completeness.*


In [4]:
search_term = "brexit"

reddit_all = reddit.subreddit("all")
results = reddit_all.search(
    query=search_term,
    time_filter="all",
    limit=None
)

search_results = []
for post in results:
  search_results.append({
      "title": post.title,
      "author": str(post.author),
      "subreddit": post.subreddit_name_prefixed,
      "score": post.score,
      "created": datetime.fromtimestamp(post.created) ,
      "permalink": post.permalink
  })
df_relevance = pd.DataFrame(search_results)
print(f"Got {len(df_relevance)} posts")
print("Latest post:", df_relevance["created"].max())
print("Earliest post:", df_relevance["created"].min())



Got 229 posts
Latest post: 2025-12-03 17:14:46
Earliest post: 2016-07-01 13:51:30


In [5]:
search_term = "brexit"

reddit_all = reddit.subreddit("all")
results = reddit_all.search(
    query=search_term,
    time_filter="all",
    sort="new",
    limit=None
)

search_results = []
for post in results:
  search_results.append({
      "title": post.title,
      "author": str(post.author),
      "subreddit": post.subreddit_name_prefixed,
      "score": post.score,
      "created": datetime.fromtimestamp(post.created) ,
      "permalink": post.permalink
  })
df_new = pd.DataFrame(search_results)
print(f"Got {len(df_new)} posts")
print("Latest post:", df_new["created"].max())
print("Earliest post:", df_new["created"].min())



Got 246 posts
Latest post: 2025-12-08 19:49:37
Earliest post: 2025-11-30 13:25:50


In [8]:
df_total = pd.concat([df_relevance, df_new], axis=0)
print(f"Found {len(df_total)} posts in both requests")
duplicated = df_total[df_total.duplicated(subset="permalink", keep="first")]
print(f"{len(duplicated)} posts were present in both requests")


Found 475 posts in both requests
2 posts were present in both requests


### ACCESSIBILITY


**OC1: Does the platform offer any type of access to non-public data (e.g., private groups, not direct messages) to approved researchers?**

*This item verifies whether, beyond the retrieval and extraction of public user-generated content data, the API permits the extraction of any data from non-public content, such as posts and comments in private groups or discussion forums, subject to the implementation of adequate data hashing measures and specific researcher approval. The assessment should confirm that the API provides such access measures, either through specific endpoints or other controlled access mechanisms.*

The platform doesn't provide access for non-public data.

**OC2: Can the requested data be extracted directly from the platform’s API response?**

*This item verifies whether the API returns structured data directly in its response, rather than only providing a redirect link to the data. Audiovisual media (e.g., images, videos, and audio) are excluded from this assessment. The assessment should check sample API responses to confirm that the requested public data is included in the returned payload itself.*


In [15]:
sr = reddit.subreddit("all")

posts = sr.new(limit=2)
posts = list(posts)

post = posts[0]

print(f"Post ID: {post.id}")
print(f"Title: {post.title}")
print(f"Subreddit: {post.subreddit_name_prefixed}")
print(f"Created At: {post.created}")
print(f"Post Body: {post.selftext}")
print(f"Post Author: {post.author.name}")

Post ID: 1p6pkip
Title: ⭐⭐⭐⭐
Subreddit: r/Monopoly_GO
Created At: 1764107237.0
Post Body: 1:1 offere
Post Author: Mammoth_Ad_1888


**OC4: Does the platform’s API offer an endpoint for extracting data from an individual publication?**

*This item verifies whether it is possible to collect data from a specific public post on the platform using a unique identifier, rather than relying on search terms or other filters. The assessment should review the API documentation and test available endpoints to confirm that an individual publication can be retrieved directly by its unique identifier.*


In [16]:
# Retrieving a post from its URL (the most common method an end user might refer to a post)

post_url = "https://www.reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/"

post = reddit.submission(url=post_url)

print(f"Post ID: {post.id}")
print(f"Title: {post.title}")
print(f"Subreddit: {post.subreddit_name_prefixed}")
print(f"Created At: {post.created}")
print(f"Post permalink: https://reddit.com{post.permalink}")
print(f"Post URL: {post.url}")
print(f"Post Body: {post.selftext}")
print(f"Post Author: {post.author.name}")

Post ID: 1oyjeib
Title: Is there much of a difference between 5600 mhz and 6000 mhz?
Subreddit: r/pcmasterrace
Created At: 1763292460.0
Post permalink: https://reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/
Post URL: https://www.reddit.com/gallery/1oyjeib
Post Body: Buying a PC soon and found a kit of ram for $230 (Australian calm down) at 5600 mhz and cl 46 and a kit of ram for $250 at 6000 mhz at cl 48. My question is: is there much of a difference in performance between these kits for gaming? 
Post Author: MrUnlucky213


In [13]:
# Retrieving a post by its ID (more programatically efficient)

post_id = "1oyjeib"

post = reddit.submission(id=post_id)

print(f"Post ID: {post.id}")
print(f"Title: {post.title}")
print(f"Subreddit: {post.subreddit_name_prefixed}")
print(f"Created At: {post.created}")
print(f"Post permalink: https://reddit.com{post.permalink}")
print(f"Post URL: {post.url}")
print(f"Post Body: {post.selftext}")
print(f"Post Author: {post.author.name}")

Post ID: 1oyjeib
Title: Is there much of a difference between 5600 mhz and 6000 mhz?
Subreddit: r/pcmasterrace
Created At: 1763292460.0
Post permalink: https://reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/
Post URL: https://www.reddit.com/gallery/1oyjeib
Post Body: Buying a PC soon and found a kit of ram for $230 (Australian calm down) at 5600 mhz and cl 46 and a kit of ram for $250 at 6000 mhz at cl 48. My question is: is there much of a difference in performance between these kits for gaming? 
Post Author: MrUnlucky213


**OC5: Does the platform’s API offer an endpoint for extracting data from an individual author?**

*This item verifies whether it is possible to collect data from public posts made by a specific author, using their username or unique identifier. The assessment should review the API documentation and test relevant endpoints to confirm that data can be retrieved directly for an individual author.*


In [ ]:
author = 'MrUnlucky213'
redditor = reddit.redditor(author)

print(f"Username: {redditor.name}")
print(f"Link Karma: {redditor.link_karma}")
print(f"Comment Karma: {redditor.comment_karma}")
print(f"Total Karma: {redditor.link_karma + redditor.comment_karma}")
print(f"Created: {redditor.created}")
print(f"Is Gold: {redditor.is_gold}")
print(f"Is Moderator: {redditor.is_mod}")
print(f"Email Verified: {redditor.verified}")


Username: MrUnlucky213
Link Karma: 2373
Comment Karma: 310
Total Karma: 2683
Created: 2024-12-30T00:08:31
Is Gold: False
Is Moderator: False
Email Verified: True


In [ ]:
# user submissions
for post in redditor.submissions.top(time_filter="all"):
    print(f"Post ID: {post.id}")
    print(f"Title: {post.title}")
    print(f"Subreddit: {post.subreddit_name_prefixed}")
    print(f"Created At: {post.created}")
    print(f"Post permalink: https://reddit.com{post.permalink}")
    print(f"Post URL: {post.url}")
    print(f"Post Body: {post.selftext}")
    print(f"Post Author: {post.author.name}")
    break

Post ID: 1oyjeib
Title: Is there much of a difference between 5600 mhz and 6000 mhz?
Subreddit: r/pcmasterrace
Created At: 1763292460.0
Post permalink: https://reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/
Post URL: https://www.reddit.com/gallery/1oyjeib
Post Body: Buying a PC soon and found a kit of ram for $230 (Australian calm down) at 5600 mhz and cl 46 and a kit of ram for $250 at 6000 mhz at cl 48. My question is: is there much of a difference in performance between these kits for gaming? 
Post Author: MrUnlucky213


**OC6: Does the platform’s API provide an endpoint for extracting data based on search terms?**

*This item verifies whether public user-generated content can be collected via individual or combined search terms, enabling the creation of datasets of posts mentioning those terms. The assessment should test search-related endpoints to confirm that queries using keywords return relevant public posts.*


In [31]:
# Use this cell to develop an example where a request for posts is made using a search term.
# Please leave the result as the output of this cell.
search_term = "cop30"

reddit_all = reddit.subreddit("all")
results = reddit_all.search(
    query=search_term,
    sort="new",
    time_filter="week",
    limit=25
)

search_results = []
for post in results:
  search_results.append({
      "title": post.title,
      "author": str(post.author),
      "subreddit": post.subreddit_name_prefixed,
      "score": post.score,
      "created": post.created,
      "permalink": post.permalink
  })

print(json.dumps(search_results, indent=2))

[
  {
    "title": "[World] - Cop30 live: Brazil aims for early agreement on \u2018big four\u2019 issues",
    "author": "AutoNewsAdmin",
    "subreddit": "r/GUARDIANauto",
    "score": 1,
    "created": 1763560600.0,
    "permalink": "/r/GUARDIANauto/comments/1p18cdv/world_cop30_live_brazil_aims_for_early_agreement/"
  },
  {
    "title": "Drag\u00e3o-on\u00e7a, monumento de artista chinesa feito para a COP30, causa revolta entre evang\u00e9licos nas redes",
    "author": "Konobajo",
    "subreddit": "r/brasil",
    "score": 4,
    "created": 1763559739.0,
    "permalink": "/r/brasil/comments/1p1809s/drag\u00e3oon\u00e7a_monumento_de_artista_chinesa_feito/"
  },
  {
    "title": "A profecia do Xereca",
    "author": "mtgsovereign",
    "subreddit": "r/CRFla",
    "score": 11,
    "created": 1763559432.0,
    "permalink": "/r/CRFla/comments/1p17w0e/a_profecia_do_xereca/"
  },
  {
    "title": "Declaration of the Peoples\u2019 Summit Towards COP30 | Rupture",
    "author": "padraigd",
 

**OC7: Does the API use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are returned in a locale-neutral format, or whether relevant locale metadata is included when neutrality is not possible. The assessment should review the API documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [28]:
sr = reddit.subreddit("all")

posts = sr.new(limit=2)
posts = list(posts)

post = posts[0]

print(f"Post ID: {post.id}")
print(f"Created At: {post.created}")
print(f"Created At UTC: {post.created_utc}")
print(f"Post Author: {post.author.name}")
print(f"Author created at: {post.author.created}")


Post ID: 1p69hn2
Created At: 1764068028.0
Created At UTC: 1764068028.0
Post Author: IDislikeNoodles
Author created at: 1528580491.0


### COMPLIANCE

**OC15: Does the platform provide a way to label content that has been generated with artificial intelligence?**

*This item verifies whether the platform automatically flags, or allows users to flag, AI-generated content, and whether this information is given in the API response. The assessment should review the documentation and test API outputs to confirm that these flags are included in the extracted data.* 


The API doesn't return any flag regarding IA-generated content.

### COMPLETENESS

**OC16: Can data from a publication’s comments be extracted using the platform’s API?**

*This item verifies whether comment data, including their content, can be extracted when available on the platform, either together with publication data or with a dedicated endpoint. The assessment should test relevant endpoints to confirm that comments are retrievable as structured data. This item does not apply to platforms that do not have commenting features.*


In [37]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
post_url = "https://www.reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/"

post = reddit.submission(url=post_url)
comments = post.comments.list()
comment = comments[0]
print(f"Post ID: {comment.parent_id}")
print(f"Comment ID: {comment.id}")
print(f"Comment Text: {comment.body}")
print(f"Comment Author: {comment.author.name}")
print(f"Comment Created At: {comment.created}")
print(f"Comment permalink: {comment.permalink}")

Post ID: t3_1oyjeib
Comment ID: np4v490
Comment Text: I know this one: the difference is about 400mhz.
Comment Author: CooterBrownJr
Comment Created At: 1763295546.0
Comment permalink: /r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/np4v490/


**OC17: Can data from temporary content be extracted through the platform’s API?**

*This item verifies whether the platform’s API provides at least one endpoint for collecting data from temporary publications (e.g., stories, ephemeral messages). The assessment should test endpoints to confirm whether this type of short-lived content can be retrieved as structured data before it expires. This item does not apply to platforms that do not have temporary content features.*


Not applicable

**OC18: Can historical data be extracted through the platform’s API?**

*This item verifies whether the API provides endpoints that allow for a specified time range, going back more than one year from the time the request is made, to collect public user-generated content data. The assessment should review test endpoints to confirm that historical data more than 12 months prior to the analysis can be retrieved.*

In [ ]:
search_term = "brexit"

reddit_all = reddit.subreddit("all")
results = reddit_all.search(
    query=search_term,
    time_filter="all",
    limit=None
)

search_results = []
for post in results:
  search_results.append({
      "title": post.title,
      "author": str(post.author),
      "subreddit": post.subreddit_name_prefixed,
      "score": post.score,
      "created": datetime.fromtimestamp(post.created) ,
      "permalink": post.permalink
  })
df_relevance = pd.DataFrame(search_results)
print(f"Got {len(df_relevance)} posts")
print("Latest post:", df_relevance["created"].max())
print("Earliest post:", df_relevance["created"].min())

Got 229 posts
Latest post: 2025-12-01 21:01:20
Earliest post: 2016-07-01 13:51:30


**OC19: Is the number of requests allowed by the API sufficient for monitoring more than 10,000 publications in 24 hours?**

*This item verifies whether data can be extracted without interruption and losses through the platform’s API for requests that accumulate more than 10,000 publications in 24 hours. The assessment should test the API to confirm that this volume of data can be collected continuously.*


In [36]:
data_dir = REDDIT_FILEPATH
stats_filename = "br-reddit-UGC-question-19-stats-2025-11-26T11-39-56-690211"

with open(f"{data_dir}/{stats_filename}", "r") as fin:
    data = json.load(fin)
    pprint.pprint(data)


{'end_time': '2025-11-26T13:39:57.434518',
 'start_time': '2025-11-26T11:39:56.690211',
 'total_data_collected': 13798}


### CONSISTENCY

**OC20: Are the results returned by the API consistently reproducible?**

*This item verifies whether data extracted via the platform’s API at any given time is consistent with other collections performed similarly, including content that has been deleted between collections. The assessment should conduct repeated test queries to confirm the reproducibility of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a test that collects data for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [ ]:
# Use this cell, or more, to develop a process that collects posts data more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
total_runs = 5 
search_term = "cop30"
for idx in range(total_runs):
    
    index = idx + 1
    data_dir = FILEPATH
    filename = f"br-reddit-UGC-question-20-run-{index}-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S-%f')}"
    reddit_all = reddit.subreddit("all")
    results = reddit_all.search(
    query=search_term,
    sort="relevance",
    time_filter="month",
    limit=25
)   
    search_results = []
    for post in results:
        response = search_results.append({
        "id": post.id,
        "title": post.title,
        "author": str(post.author),
        "subreddit": post.subreddit_name_prefixed,
        "score": post.score,
        "created": post.created,
        "permalink": post.permalink
    })
    sleep(60)
    with open(f"{data_dir}/{filename}", "w") as fout:
        json.dump(search_results, fout)

In [ ]:
files = sorted(glob.glob(f"{FILEPATH}/br-reddit-UGC-question-20*"))
dfs = [pd.read_json(f) for f in files]
id_sets = [set(df['id']) for df in dfs]

all_same_ids = all(s == id_sets[0] for s in id_sets)

print("All requests returned the same response:", all_same_ids)

All requests returned the same response: True


**OC21: Is the data returned by the platform’s API consistent with the parameters and filters used in the request?**

*This item verifies whether the data extracted through the API accurately reflects the parameters and filters specified in the request. The assessment should conduct repeated test queries to confirm the consistency of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [12]:
data_dir = FILEPATH
filename = f"br-reddit-UGC-question-21-run-1-time-filter-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S-%f')}.json"
search_term = "cop30"
reddit_all = reddit.subreddit("all")
results = reddit_all.search(
query=search_term,
time_filter="day",
limit=25
) 
search_results = []
for post in results:
    response = search_results.append({
    "id": post.id,
    "title": post.title,
    "author": str(post.author),
    "subreddit": post.subreddit_name_prefixed,
    "score": post.score,
    "created": post.created,
    "permalink": post.permalink
})

with open(f"{data_dir}/{filename}", "w") as fout:
    json.dump(search_results, fout)

In [11]:
data_dir = FILEPATH
filename = f"br-reddit-UGC-question-21-run-2-sort-new-{datetime.now().strftime('%Y-%m-%dT%H-%M-%S-%f')}"
search_term = "cop30"
reddit_all = reddit.subreddit("all")
results = reddit_all.search(
query=search_term,
sort="new",
limit=25
) 
search_results = []
for post in results:
    response = search_results.append({
    "id": post.id,
    "title": post.title,
    "author": str(post.author),
    "subreddit": post.subreddit_name_prefixed,
    "score": post.score,
    "created": post.created,
    "permalink": post.permalink
})

with open(f"{data_dir}/{filename}", "w") as fout:
    json.dump(search_results, fout)

In [19]:
# time filter: day
df = pd.read_json(f"{FILEPATH}/br-reddit-UGC-question-21-run-1-time-filter-2025-12-04T15-19-51-362008.json")
df["created_dt"] = df["created"].apply(lambda x: datetime.fromtimestamp(x))
print(f"Today: {datetime.now(tz=timezone.utc)}")
print(f"Earlist post: {df["created_dt"].min()}")
print(f"Latest post:{df["created_dt"].max()}")


Today: 2025-12-04 15:24:51.667718+00:00
Earlist post: 2025-12-03 17:18:16
Latest post:2025-12-04 14:51:55


In [22]:
# sort
df =pd.read_json(f"{FILEPATH}/br-reddit-UGC-question-21-run-2-sort-new-2025-12-04T15-19-43-328649")
df["created_dt"] = df["created"].apply(lambda x: datetime.fromtimestamp(x))
df["created_dt"]

0    2025-12-04 14:51:55
1    2025-12-04 13:11:57
2    2025-12-04 12:36:20
3    2025-12-04 12:18:27
4    2025-12-04 01:46:54
5    2025-12-04 01:40:04
6    2025-12-03 22:16:57
7    2025-12-03 18:32:32
8    2025-12-03 17:18:16
9    2025-12-03 13:45:39
10   2025-12-03 13:29:22
11   2025-12-03 13:16:45
12   2025-12-03 12:43:35
13   2025-12-03 09:43:19
14   2025-12-03 09:42:46
15   2025-12-03 09:16:41
16   2025-12-03 05:59:49
17   2025-12-02 21:55:57
18   2025-12-02 21:02:14
19   2025-12-02 20:27:30
20   2025-12-02 16:57:44
21   2025-12-02 16:42:27
22   2025-12-02 13:57:31
23   2025-12-02 12:38:55
24   2025-12-02 11:58:58
Name: created_dt, dtype: datetime64[ns]

### RELEVANCE

**OC22: Does the data extracted by the platform’s API reflect what is displayed on its user interface?**

*This item verifies whether the data returned by the API corresponds to the information displayed on the platform’s user interface at all levels of detail. The assessment should compare API responses with the user interface to confirm that key elements—such as authorship, complete content, interaction counts (e.g., comments, shares, replies), and referenced content (e.g., shares, mentions)—are fully represented.*


In [19]:
post_url = "https://www.reddit.com/r/pcmasterrace/comments/1oyjeib/is_there_much_of_a_difference_between_5600_mhz/"

post = reddit.submission(url=post_url)
print(f"Post ID: {post.id}")
print(f"Title: {post.title}")
pprint.pprint(vars(post))


Post ID: 1oyjeib
Title: Is there much of a difference between 5600 mhz and 6000 mhz?
{'_additional_fetch_params': {},
 '_comments': <praw.models.comment_forest.CommentForest object at 0x10f29c3b0>,
 '_comments_by_id': {'t1_np4pqpn': Comment(id='np4pqpn'),
                     't1_np4ptcm': Comment(id='np4ptcm'),
                     't1_np4pvi0': Comment(id='np4pvi0'),
                     't1_np4v490': Comment(id='np4v490'),
                     't1_np4vsg8': Comment(id='np4vsg8'),
                     't1_np4w61k': Comment(id='np4w61k'),
                     't1_np4weou': Comment(id='np4weou'),
                     't1_np4yrxf': Comment(id='np4yrxf'),
                     't1_np50kej': Comment(id='np50kej'),
                     't1_np52mvo': Comment(id='np52mvo'),
                     't1_np5314e': Comment(id='np5314e'),
                     't1_np537sd': Comment(id='np537sd'),
                     't1_np53e3f': Comment(id='np53e3f'),
                     't1_np53mnw': Comment(id='n

**OC23: Does the platform’s API allow for filtering data based on publisher location?**

*This item verifies whether the API supports applying location-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that data on public posts can be filtered by publisher location.*


The platform does not provide any location-based filtering parameters for posts or comments.

**OC24: Does the platform’s API allow for filtering data based on content language?**

*This item verifies whether the API allows for applying language-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by content language.*


The platform does not provide any language-based filtering parameters for posts or comments.

**OC25: Does the platform’s API allow for filtering data by specific time periods?**

*This item verifies whether the API allows applying temporal filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by custom time ranges.*


The platform does not provide a specific time filtering parameters for posts or comments. It's only possible to set the time parameter as *hour*, *day*, *week*, *month*, *year* or *all*.

### TIMELINESS

**OC26: Can data from newly published content be extracted from the platform’s API in near real time?**

*This item verifies whether the API allows the collection of data from specific content within one hour of its publication. The assessment should test the endpoint for the main content type to confirm that it allows the ready extraction of recent public posts data.*


In [20]:
sr = reddit.subreddit("all")

posts = sr.new(limit=3)
posts = list(posts)

post = posts[1]

collection_date = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
most_recent_post_date = datetime.fromtimestamp(post.created_utc)
print("Collection date:", collection_date)
print("Most recent post:", most_recent_post_date)

Collection date: 2025-11-25 21:50:17
Most recent post: 2025-11-25 21:50:07
